# Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import permutation_test
import itertools

# Data Loading

In [2]:
models = [
    "logit", # row 0
    "hotelling", # row 1
    "linear" # row 2
]

firm1 = "q"
firm2 = "pso"

shock_none_data = {
    model: {
        "avg_prices": {
            firm1: [],
            firm2: []
        },
        "deltas": {
            firm1: [],
            firm2: []
        },
        "rpdis": {
            firm1: [],
            firm2: []
        }
    } for model in models
}

shock_a_data = {
    model: {
        "avg_prices": {
            firm1: [],
            firm2: []
        },
        "deltas": {
            firm1: [],
            firm2: []
        },
        "rpdis": {
            firm1: [],
            firm2: []
        }
    } for model in models
}

shock_b_data = {
    model: {
        "avg_prices": {
            firm1: [],
            firm2: []
        },
        "deltas": {
            firm1: [],
            firm2: []
        },
        "rpdis": {
            firm1: [],
            firm2: []
        }
    } for model in models
}

shock_c_data = {
    model: {
        "avg_prices": {
            firm1: [],
            firm2: []
        },
        "deltas": {
            firm1: [],
            firm2: []
        },
        "rpdis": {
            firm1: [],
            firm2: []
        }
    } for model in models
}

for i in range(1,8):
    shock_none_df = pd.read_csv(f"../results/run_{i}/shock_none/{firm1}_vs_{firm2}.csv")
    shock_a_df = pd.read_csv(f"../results/run_{i}/shock_a/{firm1}_vs_{firm2}_schemeA.csv")
    shock_b_df = pd.read_csv(f"../results/run_{i}/shock_b/{firm1}_vs_{firm2}_schemeB.csv")
    shock_c_df = pd.read_csv(f"../results/run_{i}/shock_c/{firm1}_vs_{firm2}_schemeC.csv")

    # Map each dataframe to its corresponding data storage dictionary
    shock_mappings = [
        (shock_none_df, shock_none_data),
        (shock_a_df, shock_a_data),
        (shock_b_df, shock_b_data),
        (shock_c_df, shock_c_data)
    ]
    
    for df, data_dict in shock_mappings:
        for idx, model in enumerate(models):
            # Access the row matching the current model by its index (0, 1, or 2)
            row = df.iloc[idx]
            
            # Dynamically look up columns based on firm name capitalization (e.g., "Q" and "PSO")
            f1_upper = firm1.upper()
            f2_upper = firm2.upper()
            
            # Append Average Prices
            data_dict[model]["avg_prices"][firm1].append(row[f"{f1_upper} Avg. Prices"])
            data_dict[model]["avg_prices"][firm2].append(row[f"{f2_upper} Avg. Prices"])
            
            # Append Extra-profits Δ (deltas)
            data_dict[model]["deltas"][firm1].append(row[f"{f1_upper} Extra-profits Δ"])
            data_dict[model]["deltas"][firm2].append(row[f"{f2_upper} Extra-profits Δ"])
            
            # Append RPDI
            data_dict[model]["rpdis"][firm1].append(row[f"{f1_upper} RPDI"])
            data_dict[model]["rpdis"][firm2].append(row[f"{f2_upper} RPDI"])

In [3]:
def paired_mean_diff(x, y, axis=0):
    return np.mean(x - y, axis=axis)

# Logit

## Shock None

### Average Prices

In [4]:
array1 = np.array(shock_none_data['logit']['avg_prices'][firm1])
array2 = np.array(shock_none_data['logit']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0257
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0300, -0.0200]


### Deltas

In [5]:
array1 = np.array(shock_none_data['logit']['deltas'][firm1])
array2 = np.array(shock_none_data['logit']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0086
Exact P-value:            0.50000
95% Test-Inverted CI:     [-0.0150, 0.0350]


### RPDIs

In [6]:
array1 = np.array(shock_none_data['logit']['rpdis'][firm1])
array2 = np.array(shock_none_data['logit']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0600
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0700, -0.0500]


## Shock A

### Average Prices

In [7]:
array1 = np.array(shock_a_data['logit']['avg_prices'][firm1])
array2 = np.array(shock_a_data['logit']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0243
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0300, -0.0200]


### Deltas

In [8]:
array1 = np.array(shock_a_data['logit']['deltas'][firm1])
array2 = np.array(shock_a_data['logit']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0414
Exact P-value:            0.03125
95% Test-Inverted CI:     [-0.0800, -0.0050]


### RPDIs

In [9]:
array1 = np.array(shock_a_data['logit']['rpdis'][firm1])
array2 = np.array(shock_a_data['logit']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0786
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0850, -0.0700]


## Shock B

### Average Prices

In [10]:
array1 = np.array(shock_b_data['logit']['avg_prices'][firm1])
array2 = np.array(shock_b_data['logit']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0329
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0400, -0.0300]


### Deltas

In [11]:
array1 = np.array(shock_b_data['logit']['deltas'][firm1])
array2 = np.array(shock_b_data['logit']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0386
Exact P-value:            0.06250
95% Test-Inverted CI:     [0.0000, 0.0700]


### RPDIs

In [12]:
array1 = np.array(shock_b_data['logit']['rpdis'][firm1])
array2 = np.array(shock_b_data['logit']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0786
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0900, -0.0700]


## Shock C

### Average Prices

In [13]:
array1 = np.array(shock_c_data['logit']['avg_prices'][firm1])
array2 = np.array(shock_c_data['logit']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0229
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0300, -0.0200]


### Deltas

In [14]:
array1 = np.array(shock_c_data['logit']['deltas'][firm1])
array2 = np.array(shock_c_data['logit']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0714
Exact P-value:            0.40625
95% Test-Inverted CI:     [-0.2800, 0.1300]


### RPDIs

In [15]:
array1 = np.array(shock_c_data['logit']['rpdis'][firm1])
array2 = np.array(shock_c_data['logit']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0871
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0950, -0.0800]


# Hotelling

## Shock None

### Average Prices

In [4]:
array1 = np.array(shock_none_data['hotelling']['avg_prices'][firm1])
array2 = np.array(shock_none_data['hotelling']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-1.5, 1.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

array1 = array([1.12, 1.12, 1.12, 1.12, 1.12, 1.12, 1.12]), array2 = array([1.14, 1.14, 1.14, 1.14, 1.14, 1.14, 1.14])
  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0200
Exact P-value:            0.01562
95% Test-Inverted CI:     Outside defined search grid


### Deltas

In [17]:
array1 = np.array(shock_none_data['hotelling']['deltas'][firm1])
array2 = np.array(shock_none_data['hotelling']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0686
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.0600, 0.0800]


### RPDIs

In [18]:
array1 = np.array(shock_none_data['hotelling']['rpdis'][firm1])
array2 = np.array(shock_none_data['hotelling']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0857
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0950, -0.0800]


## Shock A

### Average Prices

In [19]:
array1 = np.array(shock_a_data['hotelling']['avg_prices'][firm1])
array2 = np.array(shock_a_data['hotelling']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0100
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0100, -0.0100]


### Deltas

In [20]:
array1 = np.array(shock_a_data['hotelling']['deltas'][firm1])
array2 = np.array(shock_a_data['hotelling']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0214
Exact P-value:            0.06250
95% Test-Inverted CI:     [0.0000, 0.0450]


### RPDIs

In [21]:
array1 = np.array(shock_a_data['hotelling']['rpdis'][firm1])
array2 = np.array(shock_a_data['hotelling']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0557
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0650, -0.0450]


## Shock B

### Average Prices

In [22]:
array1 = np.array(shock_b_data['hotelling']['avg_prices'][firm1])
array2 = np.array(shock_b_data['hotelling']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0257
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0300, -0.0200]


### Deltas

In [23]:
array1 = np.array(shock_b_data['hotelling']['deltas'][firm1])
array2 = np.array(shock_b_data['hotelling']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0871
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.0400, 0.1500]


### RPDIs

In [24]:
array1 = np.array(shock_b_data['hotelling']['rpdis'][firm1])
array2 = np.array(shock_b_data['hotelling']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.1057
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.1200, -0.0900]


## Shock C

### Average Prices

In [25]:
array1 = np.array(shock_c_data['hotelling']['avg_prices'][firm1])
array2 = np.array(shock_c_data['hotelling']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0100
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0100, -0.0100]


### Deltas

In [26]:
array1 = np.array(shock_c_data['hotelling']['deltas'][firm1])
array2 = np.array(shock_c_data['hotelling']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0486
Exact P-value:            0.45312
95% Test-Inverted CI:     [-0.1800, 0.0850]


### RPDIs

In [27]:
array1 = np.array(shock_c_data['hotelling']['rpdis'][firm1])
array2 = np.array(shock_c_data['hotelling']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0514
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.0550, -0.0500]


# Linear

## Shock None

### Average Prices

In [28]:
array1 = np.array(shock_none_data['linear']['avg_prices'][firm1])
array2 = np.array(shock_none_data['linear']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)


# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0200
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.0200, 0.0200]


### Deltas

In [29]:
array1 = np.array(shock_none_data['linear']['deltas'][firm1])
array2 = np.array(shock_none_data['linear']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

array1 = array([-0.01, -0.01, -0.01, -0.01, -0.01, -0.01, -0.01]), array2 = array([0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07])
  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0800
Exact P-value:            0.01562
95% Test-Inverted CI:     Outside defined search grid


### RPDIs

In [30]:
array1 = np.array(shock_none_data['linear']['rpdis'][firm1])
array2 = np.array(shock_none_data['linear']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.3214
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.3000, 0.3350]


## Shock A

### Average Prices

In [31]:
array1 = np.array(shock_a_data['linear']['avg_prices'][firm1])
array2 = np.array(shock_a_data['linear']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0286
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.0250, 0.0300]


### Deltas

In [32]:
array1 = np.array(shock_a_data['linear']['deltas'][firm1])
array2 = np.array(shock_a_data['linear']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0086
Exact P-value:            0.50000
95% Test-Inverted CI:     [-0.0150, 0.0300]


### RPDIs

In [33]:
array1 = np.array(shock_a_data['linear']['rpdis'][firm1])
array2 = np.array(shock_a_data['linear']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.3571
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.3500, 0.3650]


## Shock B

### Average Prices

In [34]:
array1 = np.array(shock_b_data['linear']['avg_prices'][firm1])
array2 = np.array(shock_b_data['linear']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0200
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.0200, 0.0200]


### Deltas

In [35]:
array1 = np.array(shock_b_data['linear']['deltas'][firm1])
array2 = np.array(shock_b_data['linear']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: -0.0943
Exact P-value:            0.01562
95% Test-Inverted CI:     [-0.1400, -0.0400]


### RPDIs

In [36]:
array1 = np.array(shock_b_data['linear']['rpdis'][firm1])
array2 = np.array(shock_b_data['linear']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.3057
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.2900, 0.3200]


## Shock C

### Average Prices

In [37]:
array1 = np.array(shock_c_data['linear']['avg_prices'][firm1])
array2 = np.array(shock_c_data['linear']['avg_prices'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

array1 = array([0.47, 0.47, 0.47, 0.47, 0.47, 0.47, 0.47]), array2 = array([0.44, 0.44, 0.44, 0.44, 0.44, 0.44, 0.44])
  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0300
Exact P-value:            0.01562
95% Test-Inverted CI:     Outside defined search grid


### Deltas

In [38]:
array1 = np.array(shock_c_data['linear']['deltas'][firm1])
array2 = np.array(shock_c_data['linear']['deltas'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.0700
Exact P-value:            0.29688
95% Test-Inverted CI:     [-0.0750, 0.2500]


### RPDIs

In [39]:
array1 = np.array(shock_c_data['linear']['rpdis'][firm1])
array2 = np.array(shock_c_data['linear']['rpdis'][firm2])

main_test = permutation_test(
    data=(array1, array2), 
    statistic=paired_mean_diff, 
    permutation_type='samples',     # 'samples' handles paired data structural dependencies
    alternative='two-sided', 
    n_resamples=np.inf
)

# --- 4. Compute the Test-Inverted 95% Confidence Interval ---
# We run a fine grid search to map exactly where the p-value transitions past 0.05
shifts = np.linspace(-0.5, 0.5, 10001)
ci_values = []

for shift in shifts:
    # Shift sample x by a hypothetical effect size delta
    shifted_array1 = array1 - shift
    
    perm_res = permutation_test(
        data=(shifted_array1, array2), 
        statistic=paired_mean_diff, 
        permutation_type='samples', 
        alternative='two-sided', 
        n_resamples=np.inf
    )
    
    # If p > 0.05, this shift magnitude is statistically plausible (inside the CI)
    if perm_res.pvalue > 0.05:
        ci_values.append(shift)

# Extract boundaries from the valid grid values
if len(ci_values) > 0:
    lower_bound = np.min(ci_values)
    upper_bound = np.max(ci_values)
    ci_str = f"[{lower_bound:.4f}, {upper_bound:.4f}]"
else:
    ci_str = "Outside defined search grid"
    print(f"{array1 = }, {array2 = }")


# --- 5. Report Results ---
print("=========================================")
print("  EXACT PAIRED PERMUTATION TEST REPORT   ")
print("=========================================")
print(f"Observed Mean Difference: {main_test.statistic:.4f}")
print(f"Exact P-value:            {main_test.pvalue:.5f}")
print(f"95% Test-Inverted CI:     {ci_str}")
print("=========================================")

  EXACT PAIRED PERMUTATION TEST REPORT   
Observed Mean Difference: 0.3600
Exact P-value:            0.01562
95% Test-Inverted CI:     [0.3500, 0.3700]
